In [18]:
! pip install pandas
! pip install openpyxl

In [19]:
import pandas as pd
import json
import os

In [20]:
xl_path = '../data/xl/relacao_mapas.xlsx'

In [21]:
df = pd.read_excel(xl_path, engine='openpyxl')

In [22]:
df.head()

,Número da Meta,Descrição da Meta,Status,Indicador da Legenda,Nota de Rodapé,Nota Regionalização
0,1,"Entregar 8 novos parques, para ampliar a ofert...",Regionalizada,Número de novos parques por área.,O número de parques previsto por área de subpr...,NaN
1,2,"Requalificar ou ampliar 25 parques, com a melh...",Regionalizada,Número de parques requalificados por área.,O mapa apresenta a regionalização parcial das ...,NaN
2,3,"Realizar o projeto São Paulo + Verde, que vai ...",Regionalizável,NaN,NaN,A regionalização do plantio será apresentada a...
3,4,"Alcançar 11,4% de tratamento de materiais reci...",Não regionalizável,NaN,NaN,NaN
4,5,Levar atividades de conscientização e educação...,Não regionalizável,NaN,NaN,NaN


In [23]:
df.columns

Index(['Número da Meta', 'Descrição da Meta', 'Status', 'Indicador da Legenda',
       'Nota de Rodapé', 'Nota Regionalização'],
      dtype='object')

In [24]:
mapper_columns = {
    'Número da Meta' : "meta_numero",
    "Descrição da Meta": "meta_descricao",
    "Status" : "status_regionalizacao",
    "Indicador da Legenda" : "indicador_legenda",
    "Nota de Rodapé" : "nota_rodape",
    "Nota Regionalização" : "nota_regionalizacao"
 }

In [25]:
df.rename(columns=mapper_columns, inplace=True)

In [26]:
df.head()

,meta_numero,meta_descricao,status_regionalizacao,indicador_legenda,nota_rodape,nota_regionalizacao
0,1,"Entregar 8 novos parques, para ampliar a ofert...",Regionalizada,Número de novos parques por área.,O número de parques previsto por área de subpr...,NaN
1,2,"Requalificar ou ampliar 25 parques, com a melh...",Regionalizada,Número de parques requalificados por área.,O mapa apresenta a regionalização parcial das ...,NaN
2,3,"Realizar o projeto São Paulo + Verde, que vai ...",Regionalizável,NaN,NaN,A regionalização do plantio será apresentada a...
3,4,"Alcançar 11,4% de tratamento de materiais reci...",Não regionalizável,NaN,NaN,NaN
4,5,Levar atividades de conscientização e educação...,Não regionalizável,NaN,NaN,NaN


In [27]:
df.fillna('None', inplace=True)
df.replace('None', None, inplace=True)
df.head()

,meta_numero,meta_descricao,status_regionalizacao,indicador_legenda,nota_rodape,nota_regionalizacao
0,1,"Entregar 8 novos parques, para ampliar a ofert...",Regionalizada,Número de novos parques por área.,O número de parques previsto por área de subpr...,None
1,2,"Requalificar ou ampliar 25 parques, com a melh...",Regionalizada,Número de parques requalificados por área.,O mapa apresenta a regionalização parcial das ...,None
2,3,"Realizar o projeto São Paulo + Verde, que vai ...",Regionalizável,None,None,A regionalização do plantio será apresentada a...
3,4,"Alcançar 11,4% de tratamento de materiais reci...",Não regionalizável,None,None,None
4,5,Levar atividades de conscientização e educação...,Não regionalizável,None,None,None


In [28]:
df['status_regionalizacao'] = df['status_regionalizacao'].str.lower()
df['status_regionalizacao'].unique()

array(['regionalizada', 'regionalizável', 'não regionalizável'],
      dtype=object)

In [29]:
df['meta_numero'] = df['meta_numero'].astype(int)

In [30]:
MAP_FILES = os.listdir('../data/imgs/mapas')

In [31]:
MAP_FILES[:3]

['PdM Publicação (Zoom) - Meta 95.svg',
 'PdM Publicação (Zoom) - Meta 17.svg',
 'PdM Publicação (Zoom) - Meta 24.svg']

In [32]:
def get_mapfile_meta(meta_numero:int, maps=MAP_FILES)->str:

    match  ="Meta {meta_numero}.svg"

    for map_file in maps:
        if match.format(meta_numero=meta_numero) in map_file:
            return map_file
    else:
        raise ValueError(f"Mapa para a meta {meta_numero} não encontrado")

In [33]:
parsed_metas=[]
for i, row in df.iterrows():

    meta = row.to_dict()

    meta['status_regionalizacao'] = meta['status_regionalizacao'].lower()

    if meta['status_regionalizacao']=='não regionalizável':
        meta['mapa_file'] = None
        parsed_metas.append(meta)
    elif meta['status_regionalizacao']=='regionalizável':
        meta['mapa_file'] = None
        assert meta['nota_regionalizacao'] is not None, f"Meta {meta['meta_numero']} é regionalizável, mas não tem nota de regionalização"
        parsed_metas.append(meta)
    elif meta['status_regionalizacao']=='regionalizada':
        mapa_file = get_mapfile_meta(meta['meta_numero'])
        meta['mapa_file'] = mapa_file
        parsed_metas.append(meta)
    else:
        raise ValueError(f"Status de regionalização inválido para a meta {meta['meta_numero']}: {meta['status_regionalizacao']}")
        

In [34]:
with open('../data/metas_mapas.json', 'w', encoding='utf-8') as f:
    json.dump(parsed_metas, f, ensure_ascii=False, indent=4)